In [1]:
from langchain_community.document_loaders import WebBaseLoader

/var/folders/nl/53n1sws974b46dx8jfg14w940000gn/T/ipykernel_22273/967698044.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


In [7]:
import logging
import os
import time

import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from langchain_core.documents import Document

from data.support_docs import intune_articles

load_dotenv()

logger = logging.getLogger(__name__)

_NOISE_TAGS = ["script", "style", "nav", "header", "footer", "aside", "form"]
_NOISE_CLASSES = ["toc", "feedback", "toolbar", "breadcrumb", "sidebar", "banner", "alert"]


def _fetch_content(url: str) -> str:
    headers = {"User-Agent": os.environ.get("USER_AGENT", "supportdoc-agent/1.0")}
    response = requests.get(url, headers=headers, timeout=15)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    for tag in soup.find_all(_NOISE_TAGS):
        tag.decompose()

    for tag in soup.find_all(class_=lambda c: c and any(x in " ".join(c) for x in _NOISE_CLASSES)):
        tag.decompose()

    main = soup.find("article") or soup.find("main") or soup.body
    return main.get_text(separator="\n", strip=True) if main else ""


def load_documents() -> list[Document]:
    docs = []
    failed = 0

    for article in intune_articles:
        try:
            content = _fetch_content(article["url"])
            if not content:
                raise ValueError("Empty content after parsing")

            doc = Document(
                page_content=content,
                metadata={
                    "title":    article["title"],
                    "url":      article["url"],
                    "source":   article["source"],
                    "url_type": article["url_type"],
                    "category": article["category"],
                    "platform": ",".join(article["platform"]),
                    "priority": article["priority"],
                },
            )
            docs.append(doc)
            logger.info("Loaded: %s", article["title"])
        except Exception as exc:
            failed += 1
            logger.warning("Failed to load '%s' (%s): %s", article["title"], article["url"], exc)

        time.sleep(0.5)

    logger.info("Done — %d loaded, %d failed", len(docs), failed)
    return docs


In [10]:
docs= load_documents()
docs[0].metadata

{'title': 'Troubleshoot device enrollment in Intune',
 'url': 'https://learn.microsoft.com/en-us/troubleshoot/mem/intune/device-enrollment/troubleshoot-device-enrollment-in-intune',
 'source': 'Microsoft Learn',
 'url_type': 'direct_article_url',
 'category': 'device-enrollment',
 'platform': 'windows,android,ios',
 'priority': 1}

In [11]:
print(docs[0].page_content)

Table of contents
Exit editor mode
Ask Learn
Ask Learn
Reading mode
Table of contents
Read in English
Add
Add to plan
Edit
Copy Markdown
Print
Note
Access to this page requires authorization. You can try
signing in
or
changing directories
.
Access to this page requires authorization. You can try
changing directories
.
Troubleshooting device enrollment in Intune
Feedback
Summarize this article for me
This article provides suggestions for troubleshooting
device enrollment
issues in Microsoft Intune. Browse other sections of this guide for OS-specific enrollment troubleshooting.
Initial troubleshooting steps
Before you start troubleshooting, check to make sure that you've configured Intune properly to enable enrollment. You can read about those configuration requirements in our documentation:
Set up Intune
Enroll iOS/iPadOS devices in Intune
Set up enrollment for macOS devices in Intune
Set up enrollment for Windows devices in Intune
Enroll Android devices in Intune
- No additional steps 